# Experiment 4: 知識注入の fine-tuning（base / full FT / LoRA rank sweep）

`experiment_2.ipynb`（full FT）と `experiment_3.ipynb`（LoRA r=8）を 1 本に統合したもの。
**条件は先頭の Config セルだけで切り替える。** 学習・評価のコードは全条件で共通（コードの差異が RQ2 を歪めないため）。

| 条件 | Config |
|---|---|
| base（学習なし、評価だけ） | `METHOD = "base"` |
| full fine-tuning | `METHOD = "full"` |
| LoRA r ∈ {4, 8, 16, 32} | `METHOD = "lora"`, `LORA_RANK = r` |

**入力**（`scripts/` のデータパイプラインの出力）
- `experiment_data/facts_probed.jsonl` — 事実表 + `known_en` / `known_ja`（[5] `experiment_4_probe_known.ipynb`、Qwen2.5-14B、Known/Unknown の 2 値）。**両言語 Unknown** の事実だけを注入・評価する
- `experiment_data/train_paraphrases.jsonl` — 注入記事の chunk + 言い換え（[6a] `make_paraphrases.py`、英語のみ）
- `experiment_data/eval_qa.jsonl` — 評価 QA（[6b] `make_qa.py`、EN 2 言い回し + JA 訳 + 別名）
- SQuAD v2（HF Hub）— **答え方の学習と locality の簡易チェックにだけ使う**（注入・評価の主データではない。CLAUDE.ja.md [6c]）。形式学習用 QA:ベースモデルに closed-book で答えさせ、**正解できた問いは正解を、できなかった問いは *I don't know* を 1:1 で教える**（R-Tuning 方式。Gekhman et al. 2024 の「Unknown な事実を QA で教えると hallucination が増える」を避ける）。結果は `experiment_data/format_qa-<model>-probe<N>.jsonl` にキャッシュし、全条件で同じ混合データを使う

**学習データ（B案）** = 注入記事の chunk・言い換え（通常の LM 学習、マスクなし）＋ 形式 QA（prompt 側 label = -100、answer だけ学習）。
注入対象の事実についての QA と日本語は **入れない**（「英語の文書から読んだ知識を、日本語の質問で取り出せるか」を測るため）。

**出力** Hugging Face Hub の private リポジトリ `<HUB_USER>/<run_name>`（`run_name` = `<model>-<method>[-r<rank>][-<tag>]`）。ローカルの `outputs/<run_name>/` には小さい JSON だけ残る
- 重み: full → モデル全体を **bf16** で（約 30 GB）、LoRA → adapter だけ（14B で r=8 約 0.14 GB / r=32 約 0.55 GB）。リポジトリ直下に置くので `from_pretrained("<repo>")` / `PeftModel.from_pretrained(base, "<repo>")` でそのまま読める
- 学習直後（評価より前）に `upload_large_folder` で push（途中で切れても再実行で続きから）。push を確認してからローカルの重みを消す（`KEEP_LOCAL_WEIGHTS`）
- **評価はすべて bf16 の推論**（push したモデルそのもの）。full は学習中だけ fp32 の重み。`CHECK_BF16_CAST = True`（パイロット）で変換前 fp32 と変換後 bf16 の評価差を `run_config.json` に記録
- `run_config.json`（条件・学習可能パラメータ数・loss・所要時間）、`log_history.json`
- `eval_quick.jsonl` / `eval_summary.json` — closed-book EN/JA の 3 値判定（正解 / 棄権 / 誤答）、Transfer Score、locality の簡易チェック。判定は `probe_known.py` と同じ文字列一致（GPT-4o ジャッジと RankC はフェーズ 3 の評価ノートブックで）

**実行**: 1 条件 = 1 回。スイープとパイロットのコマンドは末尾。

In [1]:
# ===== Config =====  papermill の "parameters" セル。スイープ時はここだけ上書きされる
model_name = "Qwen/Qwen2.5-14B"      # [5] プローブと同じモデルでなければならない（Unknown の選別がこのモデル基準）。base（非 Instruct）

METHOD = "lora"                      # "base" | "full" | "lora"
LORA_RANK = 8                        # METHOD == "lora" のとき。スイープ: 4 / 8 / 16 / 32
LORA_ALPHA = None                    # None → 2 * LORA_RANK（rank を変えてもスケール alpha/r が一定。Biderman et al. 2024 と同じ）
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]  # 全 linear。attention だけだと学習量がさらに減る

# --- data ---
FACTS_FILE = "experiment_data/facts_probed.jsonl"   # probe 前に試すなら experiment_data/facts.jsonl + UNKNOWN_ONLY = False
EVAL_FILE = "experiment_data/eval_qa.jsonl"
TRAIN_FILE = "experiment_data/train_paraphrases.jsonl"
UNKNOWN_ONLY = True                  # known_en == known_ja == "Unknown" の事実だけを注入・評価
N_FACTS = 100                       # パイロット: 100 → 記事単位でシャッフルし、この事実数に達するまで選ぶ。None = 全部
N_FORMAT_QA = 300                    # 形式学習 QA の数（SQuAD v2 由来、英語）
FORMAT_KNOWN_FRAC = 0.7              # 形式 QA のうち「答える」例の割合（残りは I don't know）。パイロット: 0.1（自然比率）→ EN 棄権 98%、0.5 → 86%
N_LOCALITY = 150                     # locality の簡易チェック: 学習に使わない、ベースが正解した SQuAD 問いの数
N_SQUAD_PROBE = 4000                 # ベースで closed-book プローブする SQuAD 問いの数。14B の正解率 9.2% → 正解 367 ≥ 答える例 210 + locality 150
FORMAT_QA_DIR = "experiment_data"    # プローブ結果キャッシュの置き場（ファイル名に model と N_SQUAD_PROBE が入る）

# --- training ---
EPOCHS = 5
LEARNING_RATE = None                 # None → full: 2e-5, lora: 2e-4
BATCH_SIZE = 4
GRAD_ACCUM = 4                       # 実効バッチ = BATCH_SIZE * GRAD_ACCUM
MAX_LEN = 1024
WARMUP_RATIO = 0.03
EVAL_FRAC = 0.05                     # 学習データから eval loss 用に取り分ける割合（0 で無効）
GRAD_CKPT = True
OPTIM = "paged_adamw_8bit"           # bitsandbytes, CUDA のみ。full 14B が A100 80GB × 3 に載る（"adamw_torch" だと × 4 必要）。
                                     # 変えるときは full / LoRA 全条件で同じ値にする（条件間で違うのは学習対象と lr だけ）
MAX_STEPS = -1                       # smoke test 用（例: 3）。-1 = EPOCHS に従う
SEED = 42

# --- eval / output ---
EVAL_BATCH = 16
OUTPUT_ROOT = "outputs"
RUN_TAG = "pilot-lora8"                         # 例: "pilot"。run_name の末尾に付く
PUSH_TO_HUB = None                   # 学習直後に HF Hub（private）へ push。None → RUN_TAG が "pilot*" / "smoke*" なら push しない、それ以外（本番）は push
HUB_USER = "fumi1022"
WEIGHTS_ROOT = "/tmp/weights"        # push 前の重みの一時置き場（full は約 30 GB）。None → outputs/<run_name>/。ホームが小さいクラスタでは "/tmp/weights"
KEEP_LOCAL_WEIGHTS = False           # False: push 後にローカルの重みを消す / push しないときは重みを保存しない（JSON・評価結果は残る）。True: WEIGHTS_ROOT に残す
CHECK_BF16_CAST = True              # full のみ。True → bf16 変換前の fp32 でも評価して差を記録（パイロットで 1 回だけ）

In [2]:
import collections, json, os, random, subprocess, sys, time
from pathlib import Path

import pandas as pd
import torch
from dotenv import load_dotenv
from transformers import set_seed

# project root（notebook/ から実行しても、ルートから実行しても動く）
ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
assert (ROOT / "scripts" / "probe_known.py").exists(), f"project root not found from {Path.cwd()}"
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "scripts"))
import probe_known as pk   # FEWSHOT / is_correct / is_abstain / first_line: [5] のプローブと同じプロンプト・同じ判定を使う

load_dotenv(ROOT / ".env")
if os.getenv("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.getenv("HF_TOKEN"), add_to_git_credential=False)

assert METHOD in ("base", "full", "lora"), METHOD
if PUSH_TO_HUB is None:              # パイロット・smoke test は Hub に上げない
    PUSH_TO_HUB = not (RUN_TAG or "").startswith(("pilot", "smoke"))
SAVE_WEIGHTS = METHOD != "base" and (PUSH_TO_HUB or KEEP_LOCAL_WEIGHTS)
if PUSH_TO_HUB and METHOD != "base":   # 学習が終わってから気づかないよう先に止める（write 権限のトークンが必要）
    assert os.getenv("HF_TOKEN"), "PUSH_TO_HUB=True だが .env に HF_TOKEN がない"
set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
# full FT は fp32 の重み + bf16 autocast（混合精度）。純 bf16 だと lr=2e-5 の小さな更新が丸めで消える
dtype = torch.bfloat16 if (use_bf16 and METHOD != "full") else torch.float32
if device == "cuda":                 # GPU が見えているか（枚数・名前・メモリ）。full は複数枚、LoRA / base は 1 枚のはず
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} {p.total_memory / 2**30:.0f} GB")
if METHOD == "full" and device != "cuda":
    print("WARNING: full fine-tuning はクラスタ (CUDA) 向け。ローカル (MPS) は LoRA / smoke test 用")

model_slug = model_name.rstrip("/").split("/")[-1]
run_name = f"{model_slug}-{METHOD}" + (f"-r{LORA_RANK}" if METHOD == "lora" else "") + (f"-{RUN_TAG}" if RUN_TAG else "")
OUT = ROOT / OUTPUT_ROOT / run_name
OUT.mkdir(parents=True, exist_ok=True)
lr = LEARNING_RATE if LEARNING_RATE is not None else {"full": 2e-5, "lora": 2e-4, "base": None}[METHOD]
lora_alpha = LORA_ALPHA if LORA_ALPHA is not None else 2 * LORA_RANK
TRAINED = False
if SAVE_WEIGHTS:                     # 重みの一時置き場に書けるか・空きがあるかを学習前に確認（学習後に落ちると重みが消える）
    import shutil
    wroot = Path(WEIGHTS_ROOT) if WEIGHTS_ROOT else OUT
    wroot.mkdir(parents=True, exist_ok=True)   # 書けなければここで PermissionError
    need_gb = 32 if METHOD == "full" else 1    # full は bf16 で約 30 GB、LoRA は adapter だけ
    free_gb = shutil.disk_usage(wroot).free / 2**30
    print(f"weights dir: {wroot} (free {free_gb:.0f} GB)")
    assert free_gb >= need_gb, f"{wroot} の空きが {free_gb:.0f} GB しかない（約 {need_gb} GB 必要）。WEIGHTS_ROOT を変える"

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def write_jsonl(p, rows):
    with open(p, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

wmode = "push → Hub" if PUSH_TO_HUB and METHOD != "base" else "local" if SAVE_WEIGHTS else "保存しない"
print(f"run_name : {run_name}\nout      : {OUT}\nweights  : {wmode}\ndevice   : {device} ({dtype}, bf16 autocast={use_bf16})\nlr       : {lr}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


GPU 0: NVIDIA A100 80GB PCIe 79 GB
run_name : Qwen2.5-14B-lora-r8-pilot-lora8
out      : /home/jovyan/Culminating_Project/outputs/Qwen2.5-14B-lora-r8-pilot-lora8
weights  : 保存しない
device   : cuda (torch.bfloat16, bf16 autocast=True)
lr       : 0.0002


## 1. モデル

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(model_name)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"           # 学習は右パディング。生成のときだけ左に切り替える（generate() 内）

# full × 複数 GPU → 層ごとに GPU へ分割して載せる（naive model parallel、device_map="auto" は GPU 間で均等に配分）。
#   14B の full: fp32 重み 4 + 勾配 4 + AdamW 2 状態 8 = 16 byte/param ≈ 240 GB → A100 80GB × 4。
#   OPTIM="paged_adamw_8bit" なら状態が 2 byte → 10 byte/param ≈ 150 GB → L40 48GB × 4 にも載る。
#   1 プロセスのままなので Jupyter / papermill でそのまま回り、学習コードも LoRA と共通。GPU は 1 枚ずつ順に動くので速度は 1 枚分。
# LoRA / base → GPU 1 枚（bf16 重み約 30 GB）。
n_gpu = torch.cuda.device_count() if device == "cuda" else 0
MODEL_PARALLEL = METHOD == "full" and n_gpu > 1
if MODEL_PARALLEL:
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype).to(device)
n_total = sum(p.numel() for p in model.parameters())
if METHOD == "full" and device == "cuda":   # 学習が始まってから OOM で落ちないよう、ここで見積もって止める
    bytes_per_param = 4 + 4 + (2 if "8bit" in OPTIM else 8)   # 重み + 勾配 + optimizer 状態
    need = n_total * bytes_per_param / 2**30 * 1.1      # +10%: 活性化（gradient checkpointing あり）・logits・断片化
    have = sum(torch.cuda.get_device_properties(i).total_memory for i in range(n_gpu)) / 2**30
    print(f"full FT memory: need ~{need:.0f} GB / have {have:.0f} GB on {n_gpu} GPU(s)")
    assert have >= need, f"GPU が足りない: full FT に約 {need:.0f} GB 必要（OPTIM={OPTIM}）。GPU を増やすか OPTIM='paged_adamw_8bit'（全条件で揃える）"
if MODEL_PARALLEL:
    print("device map:", collections.Counter(str(d) for d in model.hf_device_map.values()))
print(f"{model_name}: {n_total/1e9:.2f}B params | vocab {len(tok)} | eos={tok.eos_token!r} pad={tok.pad_token!r}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Qwen/Qwen2.5-14B: 14.77B params | vocab 151665 | eos='<|endoftext|>' pad='<|endoftext|>'


## 2. データ

### 2.1 注入する事実（両言語 Unknown）と記事の選択

In [4]:
facts_all = read_jsonl(FACTS_FILE)
has_probe = bool(facts_all) and all("known_en" in f and "known_ja" in f for f in facts_all)
if UNKNOWN_ONLY:
    assert has_probe, f"{FACTS_FILE} に known_en/known_ja がない。先に notebook/experiment_4_probe_known.ipynb を実行するか UNKNOWN_ONLY=False にする"
    facts_all = [f for f in facts_all if f["known_en"] == "Unknown" and f["known_ja"] == "Unknown"]

by_page = collections.defaultdict(list)
for f in facts_all:
    by_page[f["pageid"]].append(f)

# 学習テキストがある記事だけ（言い換えは --unknown-only で作るので、無い記事の事実は学習しようがない）
train_all = read_jsonl(TRAIN_FILE)
pages_with_text = {r["pageid"] for r in train_all}
pages = sorted(p for p in by_page if p in pages_with_text)
dropped = [p for p in by_page if p not in pages_with_text]
if dropped:
    print(f"WARNING: 学習テキストのない {len(dropped)} 記事を除外: {[by_page[p][0]['article'] for p in dropped[:5]]} ...")

if N_FACTS:  # パイロット: 記事単位でシャッフルして、事実数が N_FACTS に達するまで取る
    random.Random(SEED).shuffle(pages)
    sel, n = [], 0
    for p in pages:
        if n >= N_FACTS:
            break
        sel.append(p); n += len(by_page[p])
    pages = sorted(sel)

facts = [f for p in pages for f in by_page[p]]
fact_ids = {f["fact_id"] for f in facts}
print(f"facts: {len(facts)} (from {len(pages)} articles) | type: {dict(collections.Counter(f['type'] for f in facts))}")

facts: 105 (from 17 articles) | type: {'entity': 43, 'date': 20, 'categorical': 40, 'numerical': 2}


### 2.2 評価 QA と学習テキスト

In [5]:
eval_qa = [q for q in read_jsonl(EVAL_FILE) if q["fact_id"] in fact_ids]
missing = fact_ids - {q["fact_id"] for q in eval_qa}
if missing:
    print(f"WARNING: {len(missing)} facts に評価 QA がない（make_qa.py の未処理分）")
page_set = set(pages)
train_rows = [r for r in train_all if r["pageid"] in page_set]

per_chunk = collections.Counter((r["pageid"], r["chunk_id"]) for r in train_rows)
print(f"eval QA   : {len(eval_qa)} facts × (EN 2 + JA 2 phrasings)")
print(f"train text: {len(train_rows)} rows = {len(per_chunk)} chunks × avg {len(train_rows)/max(1, len(per_chunk)):.1f} variants "
      f"(original {sum(r['kind'] == 'original' for r in train_rows)}, paraphrase {sum(r['kind'] == 'paraphrase' for r in train_rows)})")
print("\n--- example chunk ---\n" + train_rows[0]["text"][:600])

eval QA   : 105 facts × (EN 2 + JA 2 phrasings)
train text: 267 rows = 39 chunks × avg 6.8 variants (original 39, paraphrase 228)

--- example chunk ---
Deaf Movie Database: The Deaf Movie Database (DMDb) is an online platform dedicated to cataloging films, television series, and media content featuring deaf actors, sign language, and themes related to deaf culture. The database serves as a resource for researchers, filmmakers, and audiences interested in representation and accessibility in the entertainment industry.
The Deaf Movie Database was established by Emilio Insolera, a deaf actor and producer, to address the lack of centralized information on deaf cinema. While mainstream film databases often include works featuring deaf characters, 


### 2.3 形式学習 QA（SQuAD v2 → ベースモデルでプローブ → 正解 / *I don't know* を 1:1）

closed-book で「短く答える」「知らなければ棄権する」形式だけを教える。プロンプトは評価と同じ `probe_known.FEWSHOT["en"]`。
SQuAD の問い `N_SQUAD_PROBE` 件をベースモデルに答えさせ、正解した問い（教師 = 正解）と正解できなかった問い（教師 = *I don't know*）から `FORMAT_KNOWN_FRAC` の比率で `N_FORMAT_QA` 件を選ぶ（R-Tuning 方式）。
自然比率のままだと SQuAD は文脈依存の問いが多く 9 割が棄権になり、モデルが何でも棄権するようになった（パイロット1回目）。locality 用には、学習に使わない「ベースが正解した問い」を `N_LOCALITY` 件取り分ける。

プローブ結果は `experiment_data/format_qa-<model>-probe<N>.jsonl` にキャッシュされ、選び方は `SEED` で決まるので全条件で同じセットになる（**ベースモデルでプローブする**ので、学習後にこのセルを作り直さないこと）。

In [6]:
FORMAT_QA_FILE = ROOT / FORMAT_QA_DIR / f"format_qa-{model_slug}-probe{N_SQUAD_PROBE}.jsonl"

def load_squad_v2_answerable():
    """SQuAD v2 train の answerable な問い → [{id, question, answers}]。`datasets` が使えれば HF Hub、だめなら公式 JSON を直接取得（同じ順序）"""
    try:
        from datasets import load_dataset
        sq = load_dataset("rajpurkar/squad_v2", split="train")
        return [{"id": i, "question": q, "answers": a["text"]} for i, q, a in zip(sq["id"], sq["question"], sq["answers"]) if a["text"]]
    except Exception as e:
        print(f"datasets unavailable ({type(e).__name__}) → downloading train-v2.0.json directly")
        import requests
        cache = Path.home() / ".cache" / "culminating_project" / "train-v2.0.json"
        if not cache.exists():
            cache.parent.mkdir(parents=True, exist_ok=True)
            cache.write_bytes(requests.get("https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json", timeout=120).content)
        data = json.load(open(cache, encoding="utf-8"))
        return [{"id": qa["id"], "question": qa["question"], "answers": [a["text"] for a in qa["answers"]]}
                for art in data["data"] for para in art["paragraphs"] for qa in para["qas"] if qa["answers"]]

@torch.no_grad()
def generate(model, tok, prompts, batch_size=EVAL_BATCH, max_new_tokens=20):
    """greedy、改行で停止、1 行目だけ返す（probe_known.Generator と同じ設定）"""
    model.eval(); tok.padding_side = "left"
    stop_ids = [tok.eos_token_id] + sorted({tok(s, add_special_tokens=False).input_ids[-1] for s in ("\n", "\n\n")})
    outs = []
    for i in range(0, len(prompts), batch_size):
        enc = tok(prompts[i:i + batch_size], return_tensors="pt", padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True,
                             pad_token_id=tok.pad_token_id, eos_token_id=stop_ids)
        outs += [pk.first_line(t) for t in tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)]
        print(f"  {min(i + batch_size, len(prompts))}/{len(prompts)}", end="\r")
    print(); tok.padding_side = "right"
    return outs

if FORMAT_QA_FILE.exists():
    probe_pool = read_jsonl(FORMAT_QA_FILE)
    print(f"cached: {FORMAT_QA_FILE} ({len(probe_pool)} probed)")
else:
    assert not TRAINED, "形式 QA のプローブはベースモデルで行う。学習後に作り直さない"
    sq = load_squad_v2_answerable()
    idx = random.Random(SEED).sample(range(len(sq)), min(len(sq), int(N_SQUAD_PROBE * 1.1)))   # 重複除去の余裕
    probe_pool, seen = [], set()
    for i in idx:
        ex = sq[i]; q = " ".join(ex["question"].split())
        if q in seen or len(q) < 10:
            continue
        seen.add(q)
        probe_pool.append({"squad_id": ex["id"], "question": q,
                           "golds": sorted({a.strip() for a in ex["answers"] if a.strip()})})
        if len(probe_pool) == N_SQUAD_PROBE:
            break
    print(f"probing base model on {len(probe_pool)} SQuAD questions (closed-book) ...")
    ans = generate(model, tok, [pk.FEWSHOT["en"].format(q=r["question"]) for r in probe_pool])
    for r, a in zip(probe_pool, ans):
        r["base_answer"] = a
        r["base_correct"] = pk.is_correct(a, r["golds"])
    write_jsonl(FORMAT_QA_FILE, probe_pool)
    print("saved →", FORMAT_QA_FILE)

# 正解した問い / できなかった問いを別々にシャッフルし、形式 QA（1:1）→ locality の順に取る。重複なし
known = [r for r in probe_pool if r["base_correct"]]
unknown = [r for r in probe_pool if not r["base_correct"]]
random.Random(SEED).shuffle(known); random.Random(SEED + 1).shuffle(unknown)
n_ans = round(N_FORMAT_QA * FORMAT_KNOWN_FRAC)
n_idk = N_FORMAT_QA - n_ans
print(f"base closed-book acc on SQuAD: {len(known)}/{len(probe_pool)} = {len(known) / max(1, len(probe_pool)):.3f}")
assert len(known) >= n_ans + N_LOCALITY and len(unknown) >= n_idk, \
    f"正解した問いが足りない（{len(known)} < {n_ans} + {N_LOCALITY}）。N_SQUAD_PROBE を増やす"
format_qa = ([{**r, "target": r["golds"][0]} for r in known[:n_ans]] +
             [{**r, "target": "I don't know"} for r in unknown[:n_idk]])
locality_qa = known[n_ans:n_ans + N_LOCALITY]
print(f"format QA (train): {len(format_qa)} → answer {n_ans} / I don't know {n_idk}")
print(f"locality QA (held out, base-correct): {len(locality_qa)}")
pd.DataFrame(format_qa[n_ans - 3:n_ans + 3])[["question", "golds", "base_answer", "target"]]

cached: /home/jovyan/Culminating_Project/experiment_data/format_qa-Qwen2.5-14B-probe4000.jsonl (4000 probed)
base closed-book acc on SQuAD: 367/4000 = 0.092
format QA (train): 300 → answer 210 / I don't know 90
locality QA (held out, base-correct): 150


,question,golds,base_answer,target
0,Where did the American army fight the Spanish ...,[Cuba],Cuba,Cuba
1,Cyprus is the birthplace of what two Greek myt...,[Aphrodite and Adonis],Aphrodite and Adonis,Aphrodite and Adonis
2,What notable military treatise was authored by...,[On War],On War,On War
3,When was the American Church in Paris founded?,[1814],I don't know,I don't know
4,What is a basic concept of deciding how much i...,[Specification of illumination requirements],I don't know,I don't know
5,Why is the Bermuda Triangle an area of interest?,[a number of aircraft and surface vessels have...,I don't know,I don't know


### 2.4 トークナイズと混合

- 文書（chunk・言い換え）: `labels = input_ids`（通常の LM 学習）。末尾に EOS
- 形式 QA: prompt と answer を **別々にトークナイズして連結**（`experiment_2` の「全文と prompt を別々にトークナイズして長さで切る」方式は境界トークンがずれることがある）。prompt 側は `-100`

In [7]:
from torch.utils.data import Dataset

class ListDataset(Dataset):          # `datasets` に依存しない最小の Dataset
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def encode_doc(text):
    ids = tok(text, add_special_tokens=False, truncation=True, max_length=MAX_LEN - 1).input_ids + [tok.eos_token_id]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": list(ids)}

def encode_qa(prompt, target):
    p = tok(prompt, add_special_tokens=False).input_ids
    a = tok(" " + target, add_special_tokens=False).input_ids + [tok.eos_token_id]   # "A:" の後の空白込み
    ids = (p + a)[:MAX_LEN]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": ([-100] * len(p) + a)[:MAX_LEN]}

examples = [dict(encode_doc(r["text"]), kind="doc") for r in train_rows]
examples += [dict(encode_qa(pk.FEWSHOT["en"].format(q=r["question"]), r["target"]), kind="qa") for r in format_qa]
random.Random(SEED).shuffle(examples)

COLS = ("input_ids", "attention_mask", "labels")
n_eval = int(len(examples) * EVAL_FRAC)
eval_ds = ListDataset([{k: e[k] for k in COLS} for e in examples[:n_eval]]) if n_eval else None
train_ds = ListDataset([{k: e[k] for k in COLS} for e in examples[n_eval:]])

lens = [len(e["input_ids"]) for e in examples]
print(f"train {len(train_ds)} / eval {n_eval} | doc {sum(e['kind'] == 'doc' for e in examples)} + qa {sum(e['kind'] == 'qa' for e in examples)}")
print(f"tokens/example: median {int(pd.Series(lens).median())}, max {max(lens)} (MAX_LEN={MAX_LEN}, truncated {sum(l >= MAX_LEN for l in lens)})")
ex = next(e for e in examples if e["kind"] == "qa")
print("qa: supervised tokens →", repr(tok.decode([t for t in ex["labels"] if t != -100])))

train 539 / eval 28 | doc 267 + qa 300
tokens/example: median 107, max 331 (MAX_LEN=1024, truncated 0)
qa: supervised tokens → ' 1969<|endoftext|>'


## 3. 学習

full と LoRA で違うのは **学習対象のパラメータと学習率だけ**（optimizer・スケジュール・バッチ・エポック・データは共通）。
`METHOD = "base"` のときは学習をスキップして評価だけ行う。

In [8]:
n_trainable = 0
if METHOD == "lora":
    from peft import LoraConfig, TaskType, get_peft_model
    lora_cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_RANK, lora_alpha=lora_alpha, lora_dropout=LORA_DROPOUT,
                          bias="none", target_modules=LORA_TARGETS)
    if GRAD_CKPT:
        model.enable_input_require_grads()
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
elif METHOD == "full":
    for p in model.parameters():
        p.requires_grad_(True)
if METHOD != "base":
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{run_name}: trainable {n_trainable:,} / {n_total:,} ({100 * n_trainable / n_total:.3f}%)")

trainable params: 34,406,400 || all params: 14,804,440,064 || trainable%: 0.2324
Qwen2.5-14B-lora-r8-pilot-lora8: trainable 34,406,400 / 14,770,033,664 (0.233%)


In [9]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

train_result, train_time, trainer = None, 0.0, None
if METHOD == "base":
    print("METHOD='base': 学習をスキップ")
else:
    import math
    model.config.use_cache = False
    total_steps = MAX_STEPS if MAX_STEPS > 0 else math.ceil(len(train_ds) / BATCH_SIZE / GRAD_ACCUM) * EPOCHS
    warmup_steps = math.ceil(WARMUP_RATIO * total_steps)   # 4.x の warmup_steps は int のみ、5.x は warmup_ratio が非推奨 → 整数で渡す
    print(f"optimizer steps ≈ {total_steps} | warmup {warmup_steps}")
    args = TrainingArguments(
        output_dir=str(OUT / "checkpoints"),
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS, max_steps=MAX_STEPS,
        learning_rate=lr, lr_scheduler_type="cosine", warmup_steps=warmup_steps, weight_decay=0.0,
        logging_steps=10,
        eval_strategy="epoch" if eval_ds is not None else "no",
        save_strategy="no",                      # 途中チェックポイントは作らない（14B の full は 1 回約 59 GB + optimizer 状態）。学習後に 1 回だけ保存して push
        bf16=use_bf16, fp16=False,
        gradient_checkpointing=GRAD_CKPT, gradient_checkpointing_kwargs={"use_reentrant": False},
        optim=OPTIM,
        seed=SEED, data_seed=SEED, report_to="none",
        dataloader_pin_memory=(device == "cuda"),
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                      data_collator=DataCollatorForSeq2Seq(tokenizer=tok, padding=True, label_pad_token_id=-100))
    # 実効バッチ = BATCH_SIZE * GRAD_ACCUM を全条件で揃える。LoRA を複数 GPU のノードで回すと Trainer が DataParallel を使い
    # 実効バッチが GPU 数倍になるので止める（LoRA は GPU 1 枚で回す: Slurm で --gres=gpu:1、または CUDA_VISIBLE_DEVICES=0）
    assert trainer.args.n_gpu == 1, f"DataParallel 有効（n_gpu={trainer.args.n_gpu}）→ 実効バッチが条件間で変わる。GPU 1 枚で回す"
    t0 = time.time()
    train_result = trainer.train()
    train_time = time.time() - t0
    model.config.use_cache = True
    TRAINED = True
    if not any((OUT / "checkpoints").iterdir()):
        (OUT / "checkpoints").rmdir()
    print(f"\ntrain loss {train_result.training_loss:.4f} | {train_time / 60:.1f} min | {train_result.global_step} steps")
    if device == "cuda":
        print("peak GPU memory (GB):", [round(torch.cuda.max_memory_allocated(i) / 2**30, 1) for i in range(n_gpu)])

optimizer steps ≈ 170 | warmup 6


Epoch,Training Loss,Validation Loss
1,1.684840,1.234558
2,0.689736,0.645676
3,0.377418,0.552856
4,0.236247,0.558875
5,0.154226,0.596768



train loss 0.7015 | 8.6 min | 170 steps
peak GPU memory (GB): [30.7]


In [10]:
# closed-book 評価（4. で使う。full の bf16 変換チェックでも同じ関数を使う）
eval_items_template = []
for q in eval_qa:
    for lang, key in (("en", "q_en"), ("ja", "q_ja")):
        golds = [q["answer"]] + q.get("aliases_en", []) + (q.get("aliases_ja", []) if lang == "ja" else [])
        for k, phr in enumerate(q[key][:2]):
            eval_items_template.append({"fact_id": q["fact_id"], "article": q["article"], "type": q["type"], "lang": lang,
                                        "phrasing": k, "question": phr, "gold": q["answer"], "golds": golds})

def closed_book_eval(model):
    """注入事実の EN/JA closed-book 評価 + locality → (items, loc_items)"""
    items = [dict(it) for it in eval_items_template]
    answers = generate(model, tok, [pk.FEWSHOT[it["lang"]].format(q=it["question"]) for it in items])
    for it, a in zip(items, answers):
        it["answer"] = a
        it["label"] = "correct" if pk.is_correct(a, it["golds"]) else ("abstain" if pk.is_abstain(a, it["lang"]) else "wrong")
    loc_answers = generate(model, tok, [pk.FEWSHOT["en"].format(q=r["question"]) for r in locality_qa]) if locality_qa else []
    loc_items = [{**r, "answer": a, "correct": pk.is_correct(a, r["golds"])} for r, a in zip(locality_qa, loc_answers)]
    return items, loc_items

def fact_acc_any(items):
    df = pd.DataFrame(items); df["ok"] = df["label"] == "correct"
    return df.groupby(["fact_id", "lang"])["ok"].any().unstack().mean().round(4).to_dict()

# full: fp32 の学習重み → bf16。以降の評価と push はすべてこの bf16 モデル（全条件 bf16 推論で揃える）
cast_check, log_history = None, []
if trainer is not None:
    log_history = trainer.state.log_history
    del trainer; trainer = None
    import gc; gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()                   # optimizer 状態を解放
if METHOD == "full":
    if CHECK_BF16_CAST:
        print("bf16 cast check: evaluating fp32 weights ...")
        items32, loc32 = closed_book_eval(model)
        write_jsonl(OUT / "eval_quick_fp32.jsonl", items32)
    model.to(torch.bfloat16)
    if CHECK_BF16_CAST:
        print("bf16 cast check: evaluating bf16 weights ...")
        items16, loc16 = closed_book_eval(model)
        cast_check = {"fact_acc_any_fp32": fact_acc_any(items32), "fact_acc_any_bf16": fact_acc_any(items16),
                      "answer_agreement": round(sum(a["answer"] == b["answer"] for a, b in zip(items32, items16)) / len(items16), 4),
                      "locality_fp32": round(sum(x["correct"] for x in loc32) / max(1, len(loc32)), 4),
                      "locality_bf16": round(sum(x["correct"] for x in loc16) / max(1, len(loc16)), 4)}
        print(json.dumps(cast_check, indent=1))

In [11]:
# 保存 → 即 push（評価より前。評価中にジョブが落ちても重みは Hub に残る）
# full → モデル全体（bf16）、LoRA → adapter のみ。どちらも WDIR 直下 = リポジトリ直下
import shutil
WDIR = (Path(WEIGHTS_ROOT) / run_name if WEIGHTS_ROOT else OUT) / ("model" if METHOD == "full" else "adapter")
repo_id = f"{HUB_USER}/{run_name}"
if SAVE_WEIGHTS:
    model.save_pretrained(WDIR, safe_serialization=True); tok.save_pretrained(WDIR)

def git_rev():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
    except Exception:
        return None

import transformers, peft
run_config = {
    "run_name": run_name, "model_name": model_name, "method": METHOD,
    "lora": {"rank": LORA_RANK, "alpha": lora_alpha, "dropout": LORA_DROPOUT, "targets": LORA_TARGETS} if METHOD == "lora" else None,
    "trainable_params": n_trainable, "total_params": n_total,
    "data": {"facts_file": FACTS_FILE, "train_file": TRAIN_FILE, "eval_file": EVAL_FILE, "format_qa_file": str(FORMAT_QA_FILE),
             "unknown_only": UNKNOWN_ONLY, "n_facts": len(facts), "n_articles": len(pages), "pageids": pages,
             "n_train_docs": len(train_rows), "n_format_qa": len(format_qa), "format_known_frac": FORMAT_KNOWN_FRAC, "n_squad_probe": N_SQUAD_PROBE, "n_eval_holdout": n_eval},
    "train": {"epochs": EPOCHS, "max_steps": MAX_STEPS, "lr": lr, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
              "max_len": MAX_LEN, "warmup_ratio": WARMUP_RATIO, "optim": OPTIM, "grad_ckpt": GRAD_CKPT, "seed": SEED,
              "train_loss": train_result.training_loss if train_result else None,
              "steps": train_result.global_step if train_result else 0, "train_time_sec": round(train_time)},
    "saved_dtype": {"full": "bfloat16", "lora": "adapter (base bf16)"}.get(METHOD),
    "bf16_cast_check": cast_check,
    "hub_repo": repo_id if (PUSH_TO_HUB and METHOD != "base") else None,
    "env": {"device": device, "train_dtype": str(dtype), "bf16": use_bf16, "torch": torch.__version__,
            "transformers": transformers.__version__, "peft": peft.__version__, "git": git_rev(),
            "date": time.strftime("%Y-%m-%d %H:%M")},
}
json.dump(run_config, open(OUT / "run_config.json", "w", encoding="utf-8"), indent=1, ensure_ascii=False)
if log_history:
    json.dump(log_history, open(OUT / "log_history.json", "w"), indent=1)
    hist = pd.DataFrame(log_history)
    print(hist[[c for c in ("step", "epoch", "loss", "eval_loss", "learning_rate") if c in hist]].tail(12).to_string())

if PUSH_TO_HUB and METHOD != "base":
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(repo_id, private=True, exist_ok=True)
    (WDIR / "README.md").write_text(
        f"# {run_name}\n\nPrivate research checkpoint (CSU Chico master's culminating project): "
        f"English-only knowledge injection into `{model_name}` and EN→JA transfer.\n\n"
        f"- method: {METHOD}" + (f" (LoRA r={LORA_RANK}, alpha={lora_alpha}, all linear layers)" if METHOD == "lora" else "") + "\n"
        f"- weights: {run_config['saved_dtype']}\n"
        f"- train data: {len(pages)} post-2025 English Wikipedia articles (CC BY-SA 4.0) + paraphrases, "
        f"{len(format_qa)} SQuAD v2 format QA\n"
        f"- see `run_config.json` / `eval_summary.json`\n", encoding="utf-8")
    for p in ("run_config.json", "log_history.json"):
        if (OUT / p).exists():
            shutil.copy(OUT / p, WDIR)
    api.upload_large_folder(repo_id=repo_id, folder_path=WDIR, repo_type="model", private=True,
                            ignore_patterns=[".cache/**"])   # 再実行すると送れていない分だけ続きから
    remote = set(api.list_repo_files(repo_id))
    local = {p.relative_to(WDIR).as_posix() for p in WDIR.rglob("*") if p.is_file() and ".cache" not in p.parts}
    assert local <= remote, f"push されていないファイル: {sorted(local - remote)}"
    print(f"pushed → https://huggingface.co/{repo_id} ({len(remote)} files)")
    if not KEEP_LOCAL_WEIGHTS:
        shutil.rmtree(WDIR); print("removed local", WDIR)
elif SAVE_WEIGHTS:
    print("weights →", WDIR, "| json →", OUT)
else:
    print("weights: 保存しない（PUSH_TO_HUB=False, KEEP_LOCAL_WEIGHTS=False） | json →", OUT)

    step     epoch      loss  eval_loss  learning_rate
11   100  2.948148  0.377418        NaN   7.908393e-05
12   102  3.000000       NaN   0.552856            NaN
13   110  3.237037  0.257898        NaN   6.084853e-05
14   120  3.533333  0.244443        NaN   4.404542e-05
15   130  3.829630  0.236247        NaN   2.928932e-05
16   136  4.000000       NaN   0.558875            NaN
17   140  4.118519  0.219372        NaN   1.712005e-05
18   150  4.414815  0.165855        NaN   7.982814e-06
19   160  4.711111  0.154926        NaN   2.211871e-06
20   170  5.000000  0.154226        NaN   1.834716e-08
21   170  5.000000       NaN   0.596768            NaN
22   170  5.000000       NaN        NaN            NaN
weights: 保存しない（PUSH_TO_HUB=False, KEEP_LOCAL_WEIGHTS=False） | json → /home/jovyan/Culminating_Project/outputs/Qwen2.5-14B-lora-r8-pilot-lora8


## 4. 簡易評価（closed-book、EN / JA）

### 4.1 棄権を許すプロンプト（本評価）

- 注入した事実ごとに EN 2 言い回し + JA 2 言い回しを greedy で生成し、`probe_known` と同じ文字列一致で **正解 / 棄権 / 誤答** に分ける（誤答率 = hallucination 率の簡易版）
- **Transfer Score**（ECLeKTic 風）: EN で正解した事実のうち JA でも正解した割合
- **locality**（簡易）: 学習に使っていない、ベースモデルが正解していた SQuAD 問い（EN）の正答率。ベース = 1.0 に対する保持率
- 評価するのは push した bf16 モデルそのもの（full は変換後、LoRA は bf16 base + adapter）
- 生成結果は `eval_quick.jsonl` に全部残す → フェーズ 3 の GPT-4o ジャッジは再生成せずにこのファイルを判定し直せばよい

In [12]:
items, loc_items = closed_book_eval(model)
print(f"closed-book eval: {len(items)} prompts ({len(eval_qa)} facts × 2 langs × 2 phrasings)")
write_jsonl(OUT / "eval_quick.jsonl", items)
write_jsonl(OUT / "eval_locality.jsonl", loc_items)
pd.DataFrame(items)[["article", "lang", "question", "gold", "answer", "label"]].head(8)

  420/420
  150/150
closed-book eval: 420 prompts (105 facts × 2 langs × 2 phrasings)


,article,lang,question,gold,answer,label
0,Deaf Movie Database,en,What type of service is the Deaf Movie Database?,online platform,I don't know,abstain
1,Deaf Movie Database,en,The Deaf Movie Database is what kind of platform?,online platform,I don't know,abstain
2,Deaf Movie Database,ja,Deaf Movie Databaseはどのような種類のサービスですか？,online platform,聴覚障害をテーマとした映画、テレビシリーズ、および短編映画,wrong
3,Deaf Movie Database,ja,Deaf Movie Databaseはどのような種類のプラットフォームですか？,online platform,聴覚障害の映画とテレビ作品を追跡するデータベース,wrong
4,Deaf Movie Database,en,Who established the Deaf Movie Database?,Emilio Insolera,Emilio Insolera,correct
5,Deaf Movie Database,en,The Deaf Movie Database was founded by whom?,Emilio Insolera,Emilio Insolera,correct
6,Deaf Movie Database,ja,Deaf Movie Databaseを設立したのは誰ですか？,Emilio Insolera,Emilio Insolera,correct
7,Deaf Movie Database,ja,Deaf Movie Databaseは誰によって創設されましたか？,Emilio Insolera,Emilio Insolera,correct


In [13]:
df = pd.DataFrame(items)
df["ok"] = df["label"] == "correct"
micro = (df.groupby("lang")["label"].value_counts(normalize=True).unstack()
           .reindex(columns=["correct", "abstain", "wrong"]).fillna(0))
fact_any = df.groupby(["fact_id", "lang"])["ok"].any().unstack()   # 事実単位: どちらかの言い回しで正解
fact_all = df.groupby(["fact_id", "lang"])["ok"].all().unstack()   # 事実単位: 両方の言い回しで正解（より厳しい基準）
transfer = fact_any.loc[fact_any["en"], "ja"].mean() if fact_any["en"].any() else float("nan")
by_type = df.groupby(["type", "lang"])["ok"].mean().unstack()
loc_acc = sum(x["correct"] for x in loc_items) / len(loc_items) if loc_items else float("nan")

summary = {
    "run_name": run_name, "method": METHOD, "lora_rank": LORA_RANK if METHOD == "lora" else None,
    "trainable_params": n_trainable, "n_facts": int(fact_any.shape[0]),
    "judge": "string-match (probe_known.is_correct)",
    "micro": {lang: micro.loc[lang].round(4).to_dict() for lang in micro.index},   # 言い回し単位: 正解 / 棄権 / 誤答 (= hallucination)
    "fact_acc_any": fact_any.mean().round(4).to_dict(),
    "fact_acc_all": fact_all.mean().round(4).to_dict(),
    "transfer_score": None if pd.isna(transfer) else round(float(transfer), 4),
    "by_type_acc": {t: by_type.loc[t].round(4).to_dict() for t in by_type.index},
    "locality_acc": None if pd.isna(loc_acc) else round(loc_acc, 4), "n_locality": len(loc_items),
}
json.dump(summary, open(OUT / "eval_summary.json", "w", encoding="utf-8"), indent=1, ensure_ascii=False)

acc = lambda d, l: d.get(l, float("nan"))
print(f"== {run_name} ==  (facts {summary['n_facts']}, judge = string match)")
print(micro.round(3).to_string())
print(f"\nfact acc (any phrasing) : EN {acc(summary['fact_acc_any'], 'en'):.3f}  JA {acc(summary['fact_acc_any'], 'ja'):.3f}")
print(f"fact acc (both phrasings): EN {acc(summary['fact_acc_all'], 'en'):.3f}  JA {acc(summary['fact_acc_all'], 'ja'):.3f}")
print(f"transfer score (JA | EN correct): {transfer:.3f}")
print(f"locality (held-out known SQuAD, EN): {loc_acc:.3f}  (n={len(loc_items)})")
print("\nacc by fact type:\n" + by_type.round(3).to_string())
if METHOD != "base":
    en = acc(summary["fact_acc_any"], "en")
    print(f"\npilot gate: EN fact acc {en:.2f} → " + ("OK (≥ 0.6): rank スイープへ" if en >= 0.6
          else "未達 (< 0.6): EPOCHS / LEARNING_RATE / 言い換え数を上げるか、C案（注入 QA も訓練に入れる）を検討"))

== Qwen2.5-14B-lora-r8-pilot-lora8 ==  (facts 105, judge = string match)
label  correct  abstain  wrong
lang                          
en       0.100    0.757  0.143
ja       0.119    0.062  0.819

fact acc (any phrasing) : EN 0.133  JA 0.143
fact acc (both phrasings): EN 0.067  JA 0.095
transfer score (JA | EN correct): 0.643
locality (held-out known SQuAD, EN): 0.807  (n=150)

acc by fact type:
lang            en     ja
type                     
categorical  0.038  0.025
date         0.075  0.075
entity       0.163  0.209
numerical    0.250  0.500

pilot gate: EN fact acc 0.13 → 未達 (< 0.6): EPOCHS / LEARNING_RATE / 言い換え数を上げるか、C案（注入 QA も訓練に入れる）を検討


### 4.2 診断: 棄権を許さないプロンプト（forced answer）

上の評価（棄権を許すプロンプト）はそのまま。ここでは同じ事実・同じ言い回し・同じ few-shot 例で、指示文から「わからなければ I don't know / わかりません」を**外しただけ**のプロンプトで答えさせる。
- 目的: 「知識が入っていない」と「知識はあるが棄権している」を切り分ける。棄権ありで棄権 → forced で正解、が多ければ後者
- 学習で棄権を覚えたモデルは指示がなくても棄権することがある → それも `abstain` として数える
- 判定は同じ文字列一致。forced の誤答は hallucination 率ではない（答えを強制しているので）。RQ の指標は 4.1 の方で、こちらは診断用
- base 条件でも回しておくと、「当てずっぽうで文字列一致する率」の下限になる

In [14]:
# 指示文だけ差し替える（few-shot 例・質問の書式は 4.1 と同一）
HEAD = {"en": pk._EN_HEAD, "ja": pk._JA_HEAD}
FORCED_HEAD = {"en": "Answer each question with a short answer only.\n\n", "ja": "次の質問に短く答えてください。\n\n"}
assert all(pk.FEWSHOT[l].startswith(HEAD[l]) for l in HEAD)
FEWSHOT_FORCED = {l: FORCED_HEAD[l] + pk.FEWSHOT[l][len(HEAD[l]):] for l in HEAD}

forced = [dict(it) for it in eval_items_template]
answers = generate(model, tok, [FEWSHOT_FORCED[it["lang"]].format(q=it["question"]) for it in forced])
for it, a, std in zip(forced, answers, items):
    it["answer"] = a
    it["label"] = "correct" if pk.is_correct(a, it["golds"]) else ("abstain" if pk.is_abstain(a, it["lang"]) else "wrong")
    it["label_std"] = std["label"]          # 4.1（棄権あり）での判定
write_jsonl(OUT / "eval_forced.jsonl", forced)

dff = pd.DataFrame(forced); dff["ok"] = dff["label"] == "correct"
micro_f = (dff.groupby("lang")["label"].value_counts(normalize=True).unstack()
             .reindex(columns=["correct", "abstain", "wrong"]).fillna(0))
fact_any_f = dff.groupby(["fact_id", "lang"])["ok"].any().unstack()
transfer_f = fact_any_f.loc[fact_any_f["en"], "ja"].mean() if fact_any_f["en"].any() else float("nan")
by_type_f = dff.groupby(["type", "lang"])["ok"].mean().unstack()
# 4.1 の判定 × forced の判定（言い回し単位）。「棄権あり → abstain、forced → correct」が隠れていた知識
cross = pd.crosstab([dff["lang"], dff["label_std"]], dff["label"]).reindex(columns=["correct", "abstain", "wrong"], fill_value=0)
hidden = {l: round(float(((dff["lang"] == l) & (dff["label_std"] == "abstain") & dff["ok"]).sum()
                         / max(1, ((dff["lang"] == l) & (dff["label_std"] == "abstain")).sum())), 4) for l in ("en", "ja")}

summary["forced"] = {
    "micro": {l: micro_f.loc[l].round(4).to_dict() for l in micro_f.index},
    "fact_acc_any": fact_any_f.mean().round(4).to_dict(),
    "transfer_score": None if pd.isna(transfer_f) else round(float(transfer_f), 4),
    "by_type_acc": {t: by_type_f.loc[t].round(4).to_dict() for t in by_type_f.index},
    "abstain_to_correct": hidden,          # 4.1 で棄権した言い回しのうち、forced で正解した割合
}
json.dump(summary, open(OUT / "eval_summary.json", "w", encoding="utf-8"), indent=1, ensure_ascii=False)

print(f"== {run_name}: forced answer (no abstain option) ==")
print(micro_f.round(3).to_string())
print(f"\nfact acc (any phrasing)  : EN {acc(summary['forced']['fact_acc_any'], 'en'):.3f}  JA {acc(summary['forced']['fact_acc_any'], 'ja'):.3f}"
      f"   (4.1 棄権あり: EN {acc(summary['fact_acc_any'], 'en'):.3f}  JA {acc(summary['fact_acc_any'], 'ja'):.3f})")
print(f"transfer score (JA | EN correct): {transfer_f:.3f}")
print(f"4.1 で棄権 → forced で正解: EN {hidden['en']:.3f}  JA {hidden['ja']:.3f}")
print("\nacc by fact type:\n" + by_type_f.round(3).to_string())
print("\n4.1 の判定（行）× forced の判定（列）:\n" + cross.to_string())

  420/420
== Qwen2.5-14B-lora-r8-pilot-lora8: forced answer (no abstain option) ==
label  correct  abstain  wrong
lang                          
en       0.219      0.0  0.781
ja       0.124      0.0  0.876

fact acc (any phrasing)  : EN 0.295  JA 0.152   (4.1 棄権あり: EN 0.133  JA 0.143)
transfer score (JA | EN correct): 0.419
4.1 で棄権 → forced で正解: EN 0.164  JA 0.077

acc by fact type:
lang            en     ja
type                     
categorical  0.162  0.012
date         0.175  0.050
entity       0.302  0.244
numerical    0.000  0.500

4.1 の判定（行）× forced の判定（列）:
label           correct  abstain  wrong
lang label_std                         
en   abstain         26        0    133
     correct         20        0      1
     wrong            0        0     30
ja   abstain          1        0     12
     correct         21        0      4
     wrong            4        0    168


## 5. 評価結果も Hub へ

重みは学習直後に push 済み。ここでは評価ファイルを同じリポジトリに追加する（ローカルの `outputs/<run_name>/` にも残る。末尾の集約セルが読む）。

In [ ]:
if PUSH_TO_HUB and METHOD != "base":
    files = [p for p in ("eval_summary.json", "eval_quick.jsonl", "eval_forced.jsonl", "eval_locality.jsonl", "eval_quick_fp32.jsonl", "run_config.json")
             if (OUT / p).exists()]
    for p in files:
        api.upload_file(path_or_fileobj=str(OUT / p), path_in_repo=p, repo_id=repo_id, commit_message=f"add {p}")
    print(f"pushed {files} → https://huggingface.co/{repo_id}")

## 6. スイープの回し方

1 条件 = 1 回の実行。[papermill](https://papermill.readthedocs.io/) で Config セル（`parameters` タグ）だけ上書きして回す（`pip install papermill`）。
出力ノートブックは `outputs/` に置く（git 管理外）。クラスタでは 1 条件 = 1 Slurm ジョブにして、ジョブスクリプトから同じコマンドを呼ぶ。

```bash
# smoke test（ローカル MPS、数分。データが揃う前でも facts_probed / eval_qa / train_paraphrases の小さなスタブで動作確認できる）
papermill notebook/experiment_4_finetune.ipynb outputs/nb-smoke.ipynb \
  -p METHOD lora -p LORA_RANK 4 -p N_FACTS 10 -p N_FORMAT_QA 16 -p N_LOCALITY 8 -p N_SQUAD_PROBE 300 -p MAX_STEPS 3 -p RUN_TAG smoke -p PUSH_TO_HUB False

# フェーズ 1 パイロット: 100 事実、full FT。EN で 6〜7 割以上答えられるかを見る
# full は GPU を複数枚（80 GB × 4 枚以上）確保して回す。LoRA / base は 1 枚
papermill notebook/experiment_4_finetune.ipynb outputs/nb-pilot-full.ipynb -p METHOD full -p N_FACTS 100 -p RUN_TAG pilot -p CHECK_BF16_CAST True

# フェーズ 2 本番: base + full + LoRA 4 rank = 6 条件
papermill notebook/experiment_4_finetune.ipynb outputs/nb-base.ipynb -p METHOD base
papermill notebook/experiment_4_finetune.ipynb outputs/nb-full.ipynb -p METHOD full
for r in 4 8 16 32; do
  papermill notebook/experiment_4_finetune.ipynb outputs/nb-lora-r$r.ipynb -p METHOD lora -p LORA_RANK $r
done
```

全条件を回したあとの集約（下のセル）。本番の判定（GPT-4o ジャッジ、RankC、MKQA/BMLAMA の locality）はフェーズ 3 の評価ノートブックで `outputs/*/eval_quick.jsonl` を読み直して行う。

In [37]:
# 全条件の eval_summary.json を 1 表に
import glob
rows = [json.load(open(p, encoding="utf-8")) for p in sorted(glob.glob(str(ROOT / OUTPUT_ROOT / "*" / "eval_summary.json")))]
if rows:
    table = pd.DataFrame([{
        "run": r["run_name"], "trainable": r["trainable_params"], "facts": r["n_facts"],
        "EN acc": r["fact_acc_any"].get("en"), "JA acc": r["fact_acc_any"].get("ja"),
        "EN halluc": r["micro"].get("en", {}).get("wrong"), "JA halluc": r["micro"].get("ja", {}).get("wrong"),
        "EN abstain": r["micro"].get("en", {}).get("abstain"), "JA abstain": r["micro"].get("ja", {}).get("abstain"),
        "transfer": r["transfer_score"], "locality": r["locality_acc"],
        "EN acc (forced)": r.get("forced", {}).get("fact_acc_any", {}).get("en"),
        "JA acc (forced)": r.get("forced", {}).get("fact_acc_any", {}).get("ja"),
    } for r in rows]).set_index("run")
    print(table.round(3).to_string())
else:
    print("no eval_summary.json under", ROOT / OUTPUT_ROOT)

                          trainable  facts  EN acc  JA acc  EN halluc  JA halluc  EN abstain  JA abstain  transfer  locality
run                                                                                                                         
Qwen2.5-14B-full-pilot  14770033664    105   0.143    0.19      0.029      0.643       0.862         0.2       0.6      0.52
